# 26_autoencoder.ipynb

**13주차 · 1교시 · 2교시** 실습 노트북

- 이론 설명과 관찰 포인트는 배포 자료(`13week/student/`)를 함께 보세요.
- 실행 환경: `%DL2026_HOME%\venv` 활성화 후 `Python (dl2026)` 커널.
- 전체 11셀. 위에서부터 순서대로 실행합니다.

## 2-1. 데이터

**셀 1** — 5주차 데이터 재사용

In [ ]:
import torch, torch.nn as nn, numpy as np
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(42)                          # 6주차 재현성 ★
DEV = "cuda" if torch.cuda.is_available() else "cpu"

tf = transforms.ToTensor()                     # [0,1] 범위 → 마지막에 Sigmoid ★
train_ds = datasets.FashionMNIST("data", train=True,  download=True, transform=tf)
test_ds  = datasets.FashionMNIST("data", train=False, download=True, transform=tf)
train_dl = DataLoader(train_ds, batch_size=256, shuffle=True)
test_dl  = DataLoader(test_ds,  batch_size=256)

CLASSES = ["티셔츠","바지","풀오버","드레스","코트","샌들","셔츠","스니커즈","가방","앵클부츠"]
x, y = next(iter(train_dl))
print("배치 :", x.shape, "| 값 범위 :", x.min().item(), "~", x.max().item())

## 2-2. 모델

**셀 2** — 오토인코더 ★ 오늘의 결과물

In [ ]:
class AutoEncoder(nn.Module):
    def __init__(self, latent=32):
        super().__init__()
        self.encoder = nn.Sequential(          # (B,784) → (B,latent)
            nn.Flatten(),
            nn.Linear(784, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, latent),            # ★ 병목
        )
        self.decoder = nn.Sequential(          # (B,latent) → (B,1,28,28)
            nn.Linear(latent, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, 784),
            nn.Sigmoid(),                      # ★ 입력이 [0,1] 이므로
            nn.Unflatten(1, (1, 28, 28)),
        )

    def forward(self, x):
        z = self.encoder(x)                    # (B, latent)
        return self.decoder(z), z              # (B,1,28,28), (B,latent)

model = AutoEncoder(latent=32).to(DEV)
out, z = model(x.to(DEV))
print("입력 :", x.shape, "→ 잠재 :", z.shape, "→ 복원 :", out.shape, " ★")
print("압축률 : 784 →", z.shape[1], f"({z.shape[1]/784*100:.1f}%)")

## 2-3. 학습 (실행 1~3분)

**셀 3** — 학습 루프 (5주차 그대로 ★)

In [ ]:
opt  = torch.optim.Adam(model.parameters(), lr=1e-3)
crit = nn.MSELoss()                            # ★ 정답 = 입력 자신
EPOCHS = 10

for ep in range(1, EPOCHS + 1):
    model.train(); tot = 0.0
    for xb, _ in train_dl:                     # ★ 레이블 _ 을 쓰지 않는다
        xb = xb.to(DEV)
        xhat, _ = model(xb)
        loss = crit(xhat, xb)                  # ★ 입력과 출력을 비교
        opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item() * xb.size(0)
    print(f"epoch {ep:2d} | train MSE {tot/len(train_ds):.5f}")

torch.save(model.state_dict(), "models/ae_fashion.pt")

> **핵심 ★★ (출제 지점)**: 학습 루프에서 **레이블 `y` 를 한 번도 쓰지 않습니다.** `for xb, _ in train_dl` 의 밑줄이 그 증거입니다. 이것이 **자기지도(self-supervised)** 이고, 생성 모델이 레이블 없이 학습되는 이유입니다.
> **관찰 포인트**: loss 가 0.06 → 0.01 근처로 내려갑니다. **0 이 되지 않는 것이 정상**입니다 — 32개 숫자로 784개를 완벽히 복원할 수는 없습니다. **그 손실이 곧 "압축의 대가"** 입니다.

## 3. 실습 2 — 원본 vs 복원 비교

**셀 4** — 눈으로 판정한다

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

model.eval()
xb, yb = next(iter(test_dl))
with torch.no_grad():
    xhat, _ = model(xb.to(DEV))
xhat = xhat.cpu()

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i in range(8):
    axes[0, i].imshow(xb[i, 0], cmap="gray");   axes[0, i].axis("off")
    axes[1, i].imshow(xhat[i, 0], cmap="gray"); axes[1, i].axis("off")
    axes[0, i].set_title(CLASSES[yb[i]], fontsize=9)
axes[0, 0].set_ylabel("원본"); axes[1, 0].set_ylabel("복원")
plt.suptitle("원본(위) vs 복원(아래) — latent=32")
plt.tight_layout(); plt.show()

> **관찰 포인트 ★**: **윤곽은 살아 있고 세부(무늬·로고·질감)는 뭉개집니다.** 32개 숫자에 담을 수 있는 것은 **"대체적인 형태"** 이지 **"세부"** 가 아니기 때문입니다. *"무엇이 남고 무엇이 사라졌는지"* 를 자기 말로 표현해 보세요.

**셀 5** — 잠재 차원을 바꾸면? (과제의 중심 ★)

In [ ]:
def train_quick(latent, epochs=5):
    m = AutoEncoder(latent).to(DEV)
    o = torch.optim.Adam(m.parameters(), lr=1e-3)
    for _ in range(epochs):
        for xb, _ in train_dl:
            xb = xb.to(DEV)
            loss = crit(m(xb)[0], xb)
            o.zero_grad(); loss.backward(); o.step()
    m.eval()
    with torch.no_grad():
        te = np.mean([crit(m(a.to(DEV))[0], a.to(DEV)).item() for a, _ in test_dl])
    return m, te

for L in [2, 8, 32]:
    _, e = train_quick(L)
    print(f"latent={L:3d} → test MSE {e:.5f}")

> **핵심 ★★ (기말 출제 지점)**: 잠재 차원을 32 → 8 → 2 로 줄이면 **복원이 점점 뭉개집니다.** 이것이 **압축과 정보 손실의 트레이드오프**를 가장 직관적으로 보여 주는 실험입니다.

## 4. 실습 3 — 잠재 공간 2D 시각화 ★

**셀 6** — latent=2 로 학습해 그대로 찍는다

In [ ]:
ae2, _ = train_quick(latent=2, epochs=8)

zs, ys = [], []
with torch.no_grad():
    for xb, yb in test_dl:
        zs.append(ae2.encoder(xb.to(DEV)).cpu()); ys.append(yb)
Z = torch.cat(zs).numpy(); Y = torch.cat(ys).numpy()
print("잠재 좌표 :", Z.shape)                   # (10000, 2)

**셀 7** — 클래스별 색으로 ★

In [ ]:
plt.figure(figsize=(8, 7))
for c in range(10):
    m_ = Y == c
    plt.scatter(Z[m_, 0], Z[m_, 1], s=3, alpha=0.5, label=CLASSES[c])
plt.legend(markerscale=4, fontsize=8, loc="best")
plt.title("잠재 공간 (latent=2) — 같은 옷 종류끼리 뭉치는가")
plt.xlabel("z₁"); plt.ylabel("z₂")
plt.tight_layout(); plt.show()

> **관찰 포인트 ★★**: **레이블을 전혀 안 썼는데 같은 종류가 뭉칩니다.** 신발류(샌들·스니커즈·앵클부츠)가 한쪽에, 상의류(티셔츠·풀오버·코트·셔츠)가 다른 쪽에 모입니다. **모델이 "옷의 종류"라는 개념을 스스로 발견한 것**입니다.
> **핵심 ★**: 이 공간이 **"의미의 지도"** 입니다. 가까운 점 = 비슷하게 생긴 옷. 10주차에 *"의미가 가까우면 벡터도 가깝다"* 고 한 **그 성질이 이미지에서도 성립**합니다.

**셀 8** — 대안: latent=32 → PCA 투영

In [ ]:
from sklearn.decomposition import PCA
zs = []
with torch.no_grad():
    for xb, _ in test_dl: zs.append(model.encoder(xb.to(DEV)).cpu())
Z32 = torch.cat(zs).numpy()
Z2  = PCA(n_components=2).fit_transform(Z32)

plt.figure(figsize=(8, 7))
for c in range(10):
    m_ = Y == c
    plt.scatter(Z2[m_, 0], Z2[m_, 1], s=3, alpha=0.5, label=CLASSES[c])
plt.legend(markerscale=4, fontsize=8); plt.title("latent=32 → PCA 2D 투영")
plt.tight_layout(); plt.show()

## 1. 실습 4 — 잠재 벡터 보간

**셀 9** — 두 이미지 사이를 8단계로 걸어간다

In [ ]:
import torch, matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Malgun Gothic"; plt.rcParams["axes.unicode_minus"] = False

model.eval()
xb, yb = next(iter(test_dl))
i, j = 0, 1                                   # 서로 다른 클래스 두 장을 고른다
print("출발 :", CLASSES[yb[i]], "→ 도착 :", CLASSES[yb[j]])

with torch.no_grad():
    z1 = model.encoder(xb[i:i+1].to(DEV))     # (1, latent)
    z2 = model.encoder(xb[j:j+1].to(DEV))

    steps = 8
    alphas = torch.linspace(0, 1, steps, device=DEV).view(-1, 1)   # (8, 1)
    zs = (1 - alphas) * z1 + alphas * z2                            # ★ 선형 보간 (8, latent)
    imgs = model.decoder(zs).cpu()                                  # (8, 1, 28, 28)

fig, axes = plt.subplots(1, steps, figsize=(2 * steps, 2.4))
for k in range(steps):
    axes[k].imshow(imgs[k, 0], cmap="gray"); axes[k].axis("off")
    axes[k].set_title(f"{alphas[k].item():.2f}", fontsize=9)
plt.suptitle("잠재 벡터 보간 — 출발에서 도착까지")
plt.tight_layout(); plt.show()

> **관찰 포인트 ★★**: 이미지가 **툭 바뀌지 않고 서서히 변합니다.** 이것이 잠재 공간이 **연속적**이라는 증거입니다. **픽셀 공간에서 같은 보간을 하면** 두 이미지가 그냥 **겹쳐 보일 뿐**입니다 — 직접 비교해 보세요.

**셀 10** — 픽셀 공간 보간과 대조 ★

In [ ]:
pix = [(1 - a) * xb[i] + a * xb[j] for a in torch.linspace(0, 1, steps)]
fig, axes = plt.subplots(1, steps, figsize=(2 * steps, 2.4))
for k in range(steps):
    axes[k].imshow(pix[k][0], cmap="gray"); axes[k].axis("off")
plt.suptitle("픽셀 공간 보간 — 두 이미지가 겹쳐 보일 뿐 ★")
plt.tight_layout(); plt.show()

> **핵심 ★**: **잠재 공간의 보간은 "의미의 변화"**, **픽셀 공간의 보간은 "투명도 합성"** 입니다. 잠재 공간이 **의미를 담고 있다**는 가장 명확한 증거입니다. (14주차 Stable Diffusion 에서 **seed 를 바꾸는 것**이 이 이야기의 연장입니다.)

## 2-2. VAE 의 발상

**셀 11** — 30초 확인: AE 로 무작위 z 를 넣으면?

In [ ]:
with torch.no_grad():
    rand_z = torch.randn(8, 32, device=DEV) * 2          # 아무 z 나
    imgs = model.decoder(rand_z).cpu()
fig, axes = plt.subplots(1, 8, figsize=(16, 2.4))
for k in range(8):
    axes[k].imshow(imgs[k, 0], cmap="gray"); axes[k].axis("off")
plt.suptitle("AE 에 무작위 z 를 넣으면 — 대개 의미 없는 얼룩 ★ (VAE 가 필요한 이유)")
plt.tight_layout(); plt.show()

> **관찰 포인트**: 알아볼 수 있는 옷이 거의 안 나옵니다. **"AE 는 복원기이지 생성기가 아니다"** 를 눈으로 보여 주는 30초입니다.